# RAG Chatbot – ShopVite
Pipeline complet : chargement → découpage → embedding → FAISS → LLM

In [1]:
# ✅ Installation des dépendances
!pip install langchain langchain-community langchain-huggingface langchain-openai faiss-cpu sentence-transformers openai

INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 1.7 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.1 MB 1.5 MB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.1 MB/s eta 0:00:00

  Attempting uninstall: openai

    Found existing installation: openai 1.82.0

   ---------------------------------------- 0/3 [openai]
    Uninstalling openai-1.82.0:
   ---------------------------------------- 0/3 [openai]
      Successfully uninstalled openai-1.82.0
   ---------------------------------------- 0/3 [openai]
   ---------------------------------------- 0/3 [openai]
   ----------------

# 1. Data Loading

In [34]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, JSONLoader
import os

def load_all_documents(data_dir="./data"):
    all_docs = []
    
    for filename in os.listdir(data_dir):
        file_path = os.path.join(data_dir, filename)
        
        # Chargement des PDF
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            all_docs.extend(loader.load())
            
        # Chargement des TXT
        elif filename.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")
            all_docs.extend(loader.load())
            
        # Chargement des JSON
        elif filename.endswith(".json"):
            loader = JSONLoader(
                file_path=file_path,
                jq_schema='.[]',
                text_content=False
            )
            all_docs.extend(loader.load())
            
    return all_docs

documents = load_all_documents()
print(f"Nombre total de documents chargés : {len(documents)}")

Nombre total de documents chargés : 8


In [35]:
# Aperçu du premier document
documents[0].page_content[:500]

'# Conditions Générales de Vente (CGV) – ShopVite\n\n## 1. Objet\n\nLes présentes Conditions Générales de Vente (ci-après « CGV ») ont pour objet de définir les droits et obligations entre la société ShopVite et toute personne physique ou morale (ci-après « le Client ») effectuant un achat de produits électroniques comme les smartphone, ordinateurs, accessoires via ses canaux de vente.\n\nToute commande implique l’acceptation pleine et entière des présentes CGV.\n\n## 2. Produits\n\nShopVite propose à la v'

# 2. Splitting

In [36]:
# ✅ CORRECTION : import depuis langchain_text_splitters (plus depuis langchain.text_splitter)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=30)
chunks = text_splitter.split_documents(documents=documents)

print(f"Nombre de chunks : {len(chunks)}")

Nombre de chunks : 17


In [37]:
# Aperçu du premier chunk
chunks[0].page_content

'# Conditions Générales de Vente (CGV) – ShopVite\n\n## 1. Objet\n\nLes présentes Conditions Générales de Vente (ci-après « CGV ») ont pour objet de définir les droits et obligations entre la société ShopVite et toute personne physique ou morale (ci-après « le Client ») effectuant un achat de produits électroniques comme les smartphone, ordinateurs, accessoires via ses canaux de vente.\n\nToute commande implique l’acceptation pleine et entière des présentes CGV.\n\n## 2. Produits\n\nShopVite propose à la vente des produits électroniques, dans la limite des stocks disponibles. Les descriptions, caractéristiques et visuels des produits sont présentés avec la plus grande exactitude possible. Toutefois, de légères variations peuvent exister.\n\n## 3. Prix'

# 3. Embedding

In [38]:
# ✅ CORRECTION : import depuis langchain_huggingface (plus depuis langchain.embeddings)
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Modèle d'embedding chargé.")

Modèle d'embedding chargé.


# 4. VectorDB (FAISS)

In [39]:
# ✅ CORRECTION : import depuis langchain_community.vectorstores (plus depuis langchain.vectorstores)
from langchain_community.vectorstores import FAISS

# Création et sauvegarde du vectorstore
vectorstore = FAISS.from_documents(chunks, embedding)
vectorstore.save_local("faiss_index")
print("Index FAISS créé et sauvegardé dans ./faiss_index")

Index FAISS créé et sauvegardé dans ./faiss_index


In [40]:
# ✅ CORRECTION : FAISS vient de langchain_community.vectorstores
# et on réutilise la variable `embedding` déjà définie ci-dessus
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)
print("Index FAISS chargé avec succès.")

Index FAISS chargé avec succès.


In [9]:
# Test de recherche par similarité
query = "avez-vous des informations sur les produits de beauté ?"
docs = vectorstore.similarity_search_with_score(query, k=2)

for doc, score in docs:
    print(f"Score: {score:.4f}")
    print(f"Contenu : {doc.page_content[:200]}")
    print("---")

Score: 0.9511
Contenu : certains produits ne peuvent faire l’objet d’un retour ni d’un remboursement, notamment : 
 Les écouteurs intra-auriculaires et tout produit similaire descellé, pour des raisons 
d’hygiène ; 
 Les p
---
Score: 1.0796
Contenu : ## 8. Garanties

Les produits vendus bénéficient des garanties légales applicables, notamment :

* Garantie légale de conformité ;
* Garantie contre les vices cachés.

En cas de produit défectueux, le
---


# 5. RAG Chatbot avec LangChain

In [46]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()  # ← charge le fichier .env

llm = ChatOpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=os.getenv("MISTRAL_API_KEY"),
    model_name="mistral-medium"
)


In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt
template = """Vous êtes l'assistant FAQ officiel de ShopVite. Votre ton est professionnel, concis et utile. 
Utilisez uniquement les segments de contexte fournis ci-dessous pour répondre à la question de l'utilisateur.

RÈGLES STRICTES :
1. Si la réponse n'est pas contenue dans le contexte, dites poliment que vous ne savez pas et proposez de contacter le support. Ne rien inventer.
2. Répondez toujours en français.

CONTEXTE :
{context}

QUESTION : 
{question}

RÉPONSE (en français) :
"""

prompt_template = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# 3. Retriever + chaîne LCEL
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {
        "context": retriever ,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("Chaîne RAG prête.")

Chaîne RAG prête.


In [33]:
# 3. Exécution d'une requête
query = "Quelle est la politique de retour pour une console ?"

result = rag_chain.invoke(query)

# 4. Affichage de la réponse
print(f"Réponse : {result}")


Réponse : D'après notre politique de retour, pour qu'une console (produit électronique) soit éligible à un retour et à un remboursement, elle doit :

1. **Être retournée dans les 30 jours** suivant la réception.
2. **Être dans son état d'origine**, non utilisée et non endommagée.
3. **Être accompagnée de tous ses accessoires** (câbles, manettes, notices, etc.) et dans **son emballage d'origine intact**.
4. **Ne pas avoir de scellé de sécurité rompu** (si applicable).

ShopVite se réserve le droit de refuser tout retour ne respectant pas ces conditions.

Pour plus de détails, consultez la section *Conditions d’éligibilité des retours* de notre politique.
